# 02 - Data Cleaning

This notebook takes the findings from `01-exploratory-data-analysis.ipynb` and turns them into a clean modeling dataset.

The goal of this notebook is **not** to train a model yet.

The goal is to:

- keep only the selected raw features
- remove rows that cannot produce a valid target
- create the `duration_days` derived target variable
- create the duration classification target
- clean text fields
- clean categorical fields
- save the cleaned dataset for preprocessing

## 02-01 Imports and notebook settings

Import the libraries needed for data cleaning.

In [58]:
from pathlib import Path

import pandas as pd

## 02-02 Load the raw dataset

Load the same raw Jira dataset used in the EDA notebook.

Adjust the path if your file is stored somewhere else.

In [ ]:
ticket_df = pd.read_csv("../jira_ticket_dataset.csv")

print(f"Raw rows: {ticket_df.shape[0]:,}")
print(f"Raw columns: {ticket_df.shape[1]:,}")

ticket_df.sample(n=3)

## 02-03 Keep selected project columns

The EDA notebook identified the following columns as useful for the first modeling version.

`created` and `resolutiondate` are used only to create the target.  
`resolutiondate` must not be used later as an input feature.

In [60]:
raw_feature_columns = [
    "summary",
    "description",
    "priority.name",
    "issuetype.name",
    "resolutiondate",
    "created",
]

# ensure all extracted features are present

missing_columns = [col for col in raw_feature_columns if col not in ticket_df.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

clean_df = ticket_df[raw_feature_columns].copy()

print(f"Rows after selecting columns: {clean_df.shape[0]:,}")
print(f"Columns after selecting columns: {clean_df.shape[1]:,}")

clean_df.head()

Rows after selecting columns: 1,149,323
Columns after selecting columns: 6


,summary,description,priority.name,issuetype.name,resolutiondate,created
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,2005-01-01 07:50:46,2005-01-01 07:47:50
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,Bug,2004-12-30 05:30:36,2004-12-25 22:50:30
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,2005-01-02 15:21:00,2005-01-01 13:52:46
3,LogHandler can only work in GlobalConfiguratio...,org.apache.axis.handlers.LogHandler in request...,Major,Bug,NaN,2005-01-02 19:13:37
4,Decoding of service is broken in org.apache.ax...,The following code assumes a lot of things:\n\...,Major,Bug,NaN,2005-01-03 03:34:52


## 02-04 Remove duplicate rows

Remove exact duplicate rows only.

In [61]:
rows_before = len(clean_df)

clean_df = clean_df.drop_duplicates().copy()

rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")

Rows before duplicate removal: 1,149,323
Rows after duplicate removal: 1,121,465
Duplicate rows removed: 27,858


## 02-05 Parse timestamp columns

Convert `created` and `resolutiondate` into datetime columns.

Invalid date values become `NaT`, which makes them easier to remove or inspect.

In [62]:
for col in ["created", "resolutiondate"]:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

## 02-06 Remove rows without valid target dates

For this first supervised classification model, rows missing `created` or `resolutiondate` cannot be used because `duration_days` cannot be calculated.

In [63]:
rows_before = len(clean_df)

clean_df = clean_df.dropna(subset=["created", "resolutiondate"]).copy()

rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print(f"Rows before removing missing target dates: {rows_before:,}")
print(f"Rows after removing missing target dates: {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")

Rows before removing missing target dates: 1,121,465
Rows after removing missing target dates: 938,537
Rows removed: 182,928


## 02-07 Create `duration_days`

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [64]:
clean_df["duration_days"] = (
    clean_df["resolutiondate"] - clean_df["created"]
).dt.total_seconds() / (60 * 60 * 24)

clean_df["duration_days"].describe()

count    938537.000000
mean        201.981867
std         517.785997
min          -0.372951
25%           1.042477
50%          11.801377
75%         110.375370
max        8001.517257
Name: duration_days, dtype: float64

The first duration summary highlights target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

### 02-07.1 Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected. 

In [66]:
display(clean_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

duration_over_365_count = (clean_df["duration_days"] > 90).sum()
print(f"Rows with duration_days > 90: {duration_over_365_count:,}")

count    938537.000000
mean        201.981867
std         517.785997
min          -0.372951
50%          11.801377
75%         110.375370
90%         617.082301
95%        1184.926491
99%        2689.743580
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 90: 253,611


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [67]:
clean_df = clean_df[clean_df["duration_days"] <= 90]

display(clean_df.shape)
clean_df.describe()

(684926, 7)

,resolutiondate,created,duration_days
count,684926,684926,684926.000000
mean,2015-11-15 17:52:39.170117,2015-11-02 07:32:01.016991,13.430997
min,2002-05-10 18:54:17,2002-05-10 18:37:38,-0.372951
25%,2012-07-11 17:19:54.500000,2012-06-29 21:32:25.750000,0.451033
50%,2016-02-20 11:22:25.500000,2016-02-08 16:03:47.500000,3.872141
75%,2019-09-04 17:15:26,2019-08-20 22:45:17.500000,17.180590
max,2025-01-27 21:10:15,2024-11-06 14:09:26,89.999618
std,NaN,NaN,20.267434


## 02-08 Remove invalid target durations

Negative durations are invalid because they mean `resolutiondate` is earlier than `created`.

In [68]:
negative_duration_count = (clean_df["duration_days"] < 0).sum()
missing_duration_count = clean_df["duration_days"].isna().sum()

rows_before = len(clean_df)

clean_df = clean_df[
    clean_df["duration_days"].notna() &
    (clean_df["duration_days"] >= 0)
].copy()

rows_after = len(clean_df)
rows_removed = rows_before - rows_after

print(f"Rows before removing invalid durations: {rows_before:,}")
print(f"Rows after removing invalid durations: {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")

Rows before removing invalid durations: 684,926
Rows after removing invalid durations: 684,908
Rows removed: 18


Tasks resolved in under 2 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [69]:
clean_df = clean_df[clean_df["duration_days"] >= (2/24)]

## 02-09 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Very Quick:** 2 hours to 1 day
- **Quick:** greater than 1 day and up to 3 days
- **Standard:** greater than 3 days and up to 7 days
- **Extended:** greater than 7 days and up to 30 days
- **Long-term:** greater than 30 days and up to 90 days


In [72]:
def duration_category(days):
    if days <= 1:
        return "Very Quick"
    if days <= 3:
        return "Quick"
    if days <= 7:
        return "Standard"
    if days <= 30:
        return "Extended"
    return "Long-term"

duration_order = ["Very Quick", "Quick", "Standard", "Extended", "Long-term"]

clean_df["duration_category"] = clean_df["duration_days"].apply(duration_category)

class_counts = clean_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (
    clean_df["duration_category"]
    .value_counts(normalize=True)
    .reindex(duration_order)
    .mul(100)
    .round(2)
)

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(clean_df.shape)
display(clean_df.describe())

duration_class_summary

(579005, 8)

,resolutiondate,created,duration_days
count,579005,579005,579005.000000
mean,2016-01-11 00:24:48.646678,2015-12-26 03:11:54.000680,15.883966
min,2002-06-03 18:35:07,2002-05-10 18:37:38,0.083333
25%,2012-09-18 00:29:07,2012-09-03 13:53:35,1.240475
50%,2016-05-13 23:50:41,2016-04-29 02:36:11,6.054028
75%,2019-11-22 07:37:16,2019-11-07 23:28:40,21.901597
max,2025-01-27 21:10:15,2024-11-06 12:43:08,89.999618
std,NaN,NaN,21.142487


,count,percent
duration_category,,
Very Quick,123476,21.33
Quick,90029,15.55
Standard,94631,16.34
Extended,160206,27.67
Long-term,110663,19.11


### 02-09.3 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [73]:
sample_columns = [
    "summary",
    "description",
    "priority.name",
    "issuetype.name",
    "duration_days",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = clean_df.loc[
        clean_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Very Quick


,summary,description,priority.name,issuetype.name,duration_days,duration_category
304387,S3 VFS driver shows size warning,The message printed in the log is:\r\n{code:ja...,Trivial,Bug,0.103542,Very Quick
20431,Use JNDI provided by the container,Right now we're using our own lightweight JNDI...,Major,Improvement,0.199444,Very Quick
896555,Unable to remove secondary ips though there ar...,Steps to reproduce :\n\n1. Have at least 1 VM ...,Critical,Bug,0.668715,Very Quick
249860,DM does not support java9+,Dependency Manager can't be built/used on java...,Major,Improvement,0.282870,Very Quick
1059743,Implement a different version of Naive Bayes P...,There has been many dependency issues with the...,Major,Improvement,0.611157,Very Quick


Quick


,summary,description,priority.name,issuetype.name,duration_days,duration_category
23876,Update for some outdated language files,Update for CommonUiLabels_ru.properties.\nUkra...,Trivial,Improvement,2.736273,Quick
330445,MatrixBlock equals,NaN,Major,Improvement,1.117789,Quick
27032,[drlvm][verifier] cleaning edges from unreacha...,"When we clean an unreachable node, all incomin...",Major,Sub-task,2.017975,Quick
968959,hbase1.3.1 regionerver Crash (bucketcache),"hbase 1.3.1 regionserver crash , configure ...",Major,Bug,2.872569,Quick
850688,Incorrect results or NPE when inserting null v...,Example:\r\n{noformat}\r\ncreate or replace te...,Major,Bug,2.313519,Quick


Standard


,summary,description,priority.name,issuetype.name,duration_days,duration_category
929996,Hovering over a border of a datagrid header cr...,Steps to reproduce:\n1. Run bug app\n2. Move m...,Major,Bug,5.154167,Standard
1125696,unix_timestamp runtime crash,in some cases \r\n\r\nselect unix_timestamp('2...,Major,Bug,6.487141,Standard
743420,Improve the Adapter API to directly yield Geom...,"Currently, the Adapter in Sedona SQL toDf meth...",Major,Improvement,3.138241,Standard
77173,TestHdfsProxy fails on 0.20,TestHdfsProxy fails with the following excepti...,Major,Bug,3.168345,Standard
265330,HTML redirections are not followed when using ...,Html redirections using meta tags are supporte...,Major,Bug,3.828900,Standard


Extended


,summary,description,priority.name,issuetype.name,duration_days,duration_category
776228,RecordsOut metric for sinks is inaccurate,"Currently, the metric is computed on the opera...",Major,Improvement,10.003588,Extended
272072,Support defining arbitrary autoscaling simulat...,In many cases where the {{bin/solr autoscaling...,Major,Improvement,14.284792,Extended
619987,uv3 migration tool should add comment about th...,The current v3 migration tool should add a com...,Minor,Improvement,16.079907,Extended
824509,Download-Link on main page is wrong,On [https://logging.apache.org/chainsaw/2.x/] ...,Major,Improvement,26.156331,Extended
91561,Implement encodeBookmarkableURL and encodeRedi...,Methods for encodeBookmarkableURL and encodeRe...,Minor,Task,21.176968,Extended


Long-term


,summary,description,priority.name,issuetype.name,duration_days,duration_category
14696,ExtensionsFilter calls addResource.parseRespon...,ExtensionsFilter does not check the content ty...,Major,Bug,34.061921,Long-term
320797,Make the HBaseMiniCluster compliant with Kerberos,Whne using MiniKDC and the minicluster in a un...,Major,Improvement,57.203241,Long-term
840936,"[C++] Writing float32 values with ""Dictionary ...",If I try to read the attached csv file with py...,Major,Bug,45.838900,Long-term
358553,Zero-sized rows in BufferedTupleStream can cau...,When running the following query on the DEBUG ...,Major,Bug,70.795475,Long-term
930261,WebService unable to read WSDL when it was abl...,Steps to reproduce:\n1. Use the attached sampl...,Major,Bug,47.926644,Long-term


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

## 02-10 Clean text fields

Clean the selected text columns lightly and create simple text-size features.

For the first model, avoid aggressive text cleaning. Technical words, punctuation, stack traces, and error names may contain useful signals.

Cleaning steps here:

- fill missing text with an empty string
- convert values to strings
- normalize whitespace
- create simple character-count and word-count features

In [85]:
text_columns = ["summary", "description"]

for col in text_columns:
    clean_col = f"{col}"
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

clean_df["summary_char_count"] = clean_df["summary"].str.len()
clean_df["description_char_count"] = clean_df["description"].str.len()

clean_df["summary_word_count"] = clean_df["summary"].str.split().str.len()
clean_df["description_word_count"] = clean_df["description"].str.split().str.len()

clean_df["has_description"] = clean_df["description"].str.len() > 0

clean_df[
    [
        "summary",
        "description",
        "summary_char_count",
        "description_char_count",
        "summary_word_count",
        "description_word_count",
        "has_description",
    ]
].head()

,summary,description,summary_char_count,description_char_count,summary_word_count,description_word_count,has_description
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,52,3445,11,206,True
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",43,2560,8,124,True
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",66,848,7,138,True
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,65,419,10,65,True
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,40,1438,7,65,True


## 02-11 Clean categorical fields

Clean only the selected categorical columns.

Cleaning steps:

- fill missing values with `"Unknown"`
- normalize whitespace
- keep category names as readable strings

In [86]:
categorical_columns = [
    "priority.name",
    "issuetype.name",
]

for col in categorical_columns:
    clean_col = col.replace(".", "_")
    clean_df[clean_col] = (
        clean_df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

clean_df[
    [
        "priority_name",
        "issuetype_name",
    ]
].head()

,priority_name,issuetype_name
1,Blocker,Bug
2,Critical,Bug
5,Major,Improvement
6,Major,Bug
8,Major,Bug


## 02-12 Check cleaned missing values

Inspect the cleaned modeling fields before saving.

In [87]:
cleaned_model_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "has_description",
    "duration_days",
    "duration_category",
]

cleaned_missing_summary = pd.DataFrame({
    "missing_count": clean_df[cleaned_model_columns].isna().sum(),
    "missing_percent": clean_df[cleaned_model_columns].isna().mean().mul(100).round(2),
})

cleaned_missing_summary.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
summary,0,0.0
description,0,0.0
priority_name,0,0.0
issuetype_name,0,0.0
summary_char_count,0,0.0
summary_word_count,0,0.0
description_char_count,0,0.0
description_word_count,0,0.0
has_description,0,0.0
duration_days,0,0.0


## 02-13 Create final cleaned modeling dataframe

Create a final dataframe containing only the columns needed for preprocessing and modeling.

Important leakage rule:

- `resolutiondate` is excluded from the final modeling dataframe.
- `created` is excluded from the first modeling dataframe.
- `duration_days` is kept for analysis and target traceability.
- `duration_category` is the classification target.

In [88]:
model_df = clean_df[cleaned_model_columns].copy()
model_df_sample = model_df.sample(n=100, random_state=42)

print(f"Final cleaned modeling rows: {model_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {model_df.shape[1]:,}")

model_df.head()

Final cleaned modeling rows: 579,005
Final cleaned modeling columns: 11


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,has_description,duration_days,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,True,4.277847,Standard
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,True,1.061273,Quick
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,True,7.366100,Extended
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,True,5.693461,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,True,32.141562,Long-term


## 02-14 Save cleaned dataset

Save the cleaned dataset for the preprocessing notebook.

Parquet is preferred when available because it preserves data types and is usually faster than CSV.

A CSV fallback is also included.

In [89]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "jira_issues_cleaned.csv"
csv_sample_path = output_dir / "jira_issues_cleaned_sample.csv"



model_df.to_csv(csv_path)
model_df_sample.to_csv(csv_sample_path)
print(f"Saved CSV file to: {csv_path}")
print(f"Saved Sample CSV file to: {csv_sample_path}")


Saved CSV file to: ..\data\processed\jira_issues_cleaned.csv
Saved Sample CSV file to: ..\data\processed\jira_issues_cleaned_sample.csv


## 02-15 Cleaning summary

This notebook produced a clean dataset for preprocessing.

Cleaning actions completed:

- selected project-relevant features
- removed duplicate rows
- parsed timestamps
- removed rows missing `created` or `resolutiondate`
- created `duration_days`
- removed invalid negative durations
- created `duration_category`
- cleaned text fields
- created simple text-size features
- cleaned categorical fields
- saved the cleaned modeling dataset